# 📈 Getting Constant Maturity Yields From FRED
<br>

<div style="display: flex; flex-wrap: wrap; align-items: center; gap: 15px; margin-bottom: 25px; padding-bottom: 15px; border-bottom: 1px solid #eaeaea;">
  
  <a href="https://colab.research.google.com/github/PatrickJHess/Volume-Four-Chapter-One/blob/master/colab/Colab_Getting_Constant_Maturity_Yields_From_FRED.ipynb" target="_blank" style="display: flex; align-items: center;">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="height: 28px; margin: 0;">
  </a>

  <a href="https://mybinder.org/v2/gh/PatrickJHess/Volume-Four-Chapter-One/master?urlpath=lab/tree/notebooks/Getting_Constant_Maturity_Yields_From_FRED.ipynb" target="_blank" style="background-color: #f5a252; color: white; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">🚀</span> Launch Live in Binder
  </a>

  <a href="https://patrickjhess.github.io/Volume-Four-Chapter-One/" style="background-color: #f1f3f4; color: #3c4043; border: 1px solid #dadce0; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">⬅️</span> Return to Main Book
  </a>
</div>

This notebook puts our secure key management and data pipeline to FRED to work. Before we dive in, you need to have some familiarity with FRED series IDs and how to extract them. As you will see, the `get_series` method of `FredReader` deftly handles incorrect series IDs and can be used to sort out the correct ones.

Start off by watching this short video on the FRED database and its series IDs


[![Getting Series IDs From Fred](https://img.youtube.com/vi/ZP1hKFRZAz4/0.jpg)](https://youtu.be/ZP1hKFRZAz4)

:::{important} 🤔 Notebook Setup: Why the "Try/Except" Imports?
:class: dropdown

**The Goal:**
To ensure this notebook runs perfectly whether you are using **Google Colab**, a local **Jupyter instance**, or a remote server without you having to manually install software.

* **External Libraries:** NumPy and Pandas are the "heavy hitters" for data. They aren't always installed by default.
* **The `try/except` Logic:** This is a safety net.
    1. We **try** to import the library.
    2. If it fails (because it's not installed), the **except** block triggers a `%pip install` to download it automatically.
* **Aliasing (`as np`):** We rename `numpy` to `np` to save keystrokes. In professional finance code, `np` and `pd` are the universal shorthand.
  
:::

## 🛠️ Preparing the Notebook

<details>
<summary><b>👉 Click to Expand: 📦 Importing Libraries, Modules, and Functions</b></summary>

As a best practice, we always begin by importing our necessary dependencies in the very first code cell. Notice how lightweight our imports are here: just the `pandas` library. ✨

This simplicity is a direct result of using the `financial_quant` package, which handles the heavy lifting behind the scenes. 🏗️ By keeping complicated setup details out of sight, we ensure the spotlight remains focused exactly where it belongs—on the core analysis. 🎯

```python
try:
    import pandas as pd
except:
    %pip -q install pandas
    import pandas as pd
```
**👀 Keep an eye out**: As we progress, pay attention to how the `financial_quant` package is imported as `fq`, and how every reference to its functions begins with fq.. 💡 This follows the exact same standard practice we demonstrated in Chapter One with NumPy (np) and Pandas (pd).

</details>


In [1]:
try:
    import pandas as pd
except:
    !pip -q install pandas
    import pandas as pd

## 📦 Getting Functions from financial_quant package

<details>
<summary><b style="font-size:1.2em; color: #1976d2; cursor: pointer;">🔌 Professional Packaging: How GitHub Installations Work</b></summary>
<br>
<p><b>The Logic:</b><br>
Usually, Python looks for modules as <code>.py</code> files on your hard drive. Here, we are "tricking" Python into treating a string of text from a URL as a live library.</p>

<p><b>The Workflow:</b></p>
<ol>
<li><b>Fetch & Build:</b> The <code>%pip install git+https://...</code> command tells your Jupyter environment to clone the repository from GitHub and install the <code>financial_quant</code> package directly into your system's site-packages directory.</li>
<li><b>Import:</b> <code>import financial_quant as fq</code> loads the package into your notebook's memory and assigns it the quick alias <code>fq</code>.</li>
<li><b>Routing:</b> Behind the scenes, a special file called <code>__init__.py</code> acts as the package's "front door." It automatically gathers complex tools from deeply nested folders (like our fixed-income models and chart visualizers) and serves them up directly to the surface.</li>
<li><b>Execute:</b> You don't have to worry about where the files live. You just type <code>fq.one_y_axis() or fq.calc_ytm()</code>, and Python immediately knows where to route the request.</li>
</ol>

<p><b>Why do this?</b><br>
This is exactly how professional software engineering teams manage and distribute code. It keeps your notebooks incredibly clean, ensures everyone is using the exact same version of the math models, and guarantees your code is 100% portable to any cloud environment</p>
</details>

In [2]:
!pip install -q git+https://github.com/PatrickJHess/quant_repo.git 2> /dev/null
import financial_quant as fq

## ✅ Authenticate your FRED API key

In [3]:
fq.secure_key_setup("fred_key")

## 🏦 🆔 Series IDs for constant maturity yields and the secured overnight funding rate SOFR

Generate a potential list of Series IDs for constant maturity yields as `f` strings..

*   **Monthly Series**: all monthly maturities between one and eleven months

```
monthly_ids = [f"DGS{i}MO" for i in range(1, 12)]
```


*   **Yearly Series**: all years between one and thirty years

```
yearly_ids = [f"DGS{i}" for i in range(1, 31)]
```

Secured overnight Id is 'SOFR'.  The list of all IDs is series_id.

In [4]:
# Generate months 1-11 and years 1-30 programmatically
monthly_ids = [f"DGS{i}MO" for i in range(1, 12)]
yearly_ids = [f"DGS{i}" for i in range(1, 31)]

# Combine everything together with SOFR
series_ids = ['sofr'] + monthly_ids + yearly_ids

## 🔗  Accessing constant maturity yields for May 2026 with `FredReader`.

The `get_series` method of `FredReader`

In [7]:
# create an instance of FredReader
fred_data=fq.FredReader()

# call the method for the class
yield_data=fred_data.get_series(series_ids,start_date='2026-05-01')
display(yield_data)

📂 FRED Cache anchored at: /home/pat/fred
✅ Key loaded from local environment ('fred_key')

☁️--- Processing sofr ---
🕒 Metadata is fresh (3 days old).
✅ Loaded sofr from local cache.

☁️--- Processing DGS1MO ---
🕒 Metadata is fresh (3 days old).
✅ Loaded DGS1MO from local cache.

☁️--- Processing DGS2MO ---
🆕 First run for DGS2MO. Initializing metadata...
❌ API REJECTED for 'DGS2MO': Bad Request.  The series does not exist.
⚠️ Skipping DGS2MO: No data was returned.

☁️--- Processing DGS3MO ---
🕒 Metadata is fresh (3 days old).
✅ Loaded DGS3MO from local cache.

☁️--- Processing DGS4MO ---
🆕 First run for DGS4MO. Initializing metadata...
❌ API REJECTED for 'DGS4MO': Bad Request.  The series does not exist.
⚠️ Skipping DGS4MO: No data was returned.

☁️--- Processing DGS5MO ---
🆕 First run for DGS5MO. Initializing metadata...
❌ API REJECTED for 'DGS5MO': Bad Request.  The series does not exist.
⚠️ Skipping DGS5MO: No data was returned.

☁️--- Processing DGS6MO ---
🕒 Metadata is fresh (3 d

,sofr,DGS1MO,DGS3MO,DGS6MO,DGS1,DGS2,DGS3,DGS5,DGS7,DGS10,DGS20,DGS30
DATE,,,,,,,,,,,,
2026-05-01,3.64,3.71,3.68,3.71,3.73,3.88,3.91,4.02,4.20,4.39,4.96,4.97
2026-05-04,3.63,3.71,3.70,3.76,3.78,3.95,3.98,4.08,4.26,4.45,5.01,5.02
2026-05-05,3.62,3.70,3.69,3.75,3.77,3.93,3.97,4.08,4.25,4.43,4.98,4.98
2026-05-06,3.61,3.70,3.69,3.74,3.73,3.87,3.89,3.99,4.17,4.36,4.92,4.94
2026-05-07,3.60,3.72,3.69,3.74,3.76,3.92,3.94,4.04,4.22,4.41,4.96,4.97
2026-05-08,3.60,3.71,3.69,3.74,3.75,3.90,3.92,4.02,4.19,4.38,4.93,4.95
2026-05-11,3.60,3.71,3.70,3.77,3.79,3.95,3.96,4.07,4.24,4.42,4.97,4.98
2026-05-12,3.60,3.71,3.70,3.77,3.80,4.00,4.01,4.12,4.29,4.46,5.02,5.03
2026-05-13,3.59,3.71,3.69,3.77,3.79,3.98,4.00,4.12,4.28,4.46,5.03,5.03


 ### ✍️ FRED Data Challenge
>


*   Access Overnight Secured Funding Rate And Par Yield For Thirty Year Maturity between January 1, 2026 and May 20, 2026.
*   Did you access series from cache or FRED?


💡 Tip: Use the first and last column heads of `yield_data` as series IDs.
---


<details>
<summary><b>✅ Example Of Solution</b></summary>


**Example of Code**
```python
# assuming imports of this notebook
fred_data=fq.FredReader()
series_ids=[yield_data.columns[0],yield_data.columns[-1]]
fred_data.get_series(series_ids,start_date='2026-01-01',end_date='2026-05-20')
```
</details>

---
<details>
<summary style="cursor: pointer; color: #2196f3; font-weight: bold;">👉 Click here to reveal the answers</summary>
<div style="margin-top: 10px; padding: 10px; border-left: 3px solid #2196f3; background-color: #f9f9f9;">
The previous request for FRED data specified May 1, 2026 as the start date.  The data request can not be completed with cache.  The cache for the two series `sofr` and `DGS30` is updated.
</details>
